# BeeAI Learning Assistant — Llama 3.2 3B (Q4_K_M) with llama.cpp

This cleaned-up Google Colab notebook adapts the uploaded BeeAI example to use **Llama 3.2 3B Instruct (Q4_K_M)** served locally by **llama.cpp**. It removes the Ollama server, ChatOllama-style configuration, and Qwen model.

## Before you run
- Use a Colab runtime with enough disk space and RAM for the GGUF model.
- A GPU is recommended. The build cell attempts a CUDA build when a CUDA GPU is available; otherwise it builds CPU-only.
- The initial build and model download may take several minutes.
- This notebook uses BeeAI's `ChatModel` OpenAI backend against llama.cpp's OpenAI-compatible API. If your installed BeeAI version changes its backend API, see the troubleshooting note at the end.


## Cell 1

In [ ]:
# Install system build tools and Python dependencies.
!sudo apt-get update -qq
!sudo apt-get install -y -qq cmake build-essential git

%pip install -q beeai-framework huggingface_hub requests

## Cell 2

In [ ]:
import os
import sys
import time
import shutil
import subprocess
from pathlib import Path

import requests
import beeai_framework

print("Python:", sys.version)
print("BeeAI installed successfully!")
print("CUDA tool available:", shutil.which("nvidia-smi") is not None)

## Cell 3

In [ ]:
# Build llama.cpp. A fresh Colab runtime may take several minutes here.
HAS_CUDA = shutil.which("nvidia-smi") is not None
if HAS_CUDA:
    gpu_check = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    HAS_CUDA = gpu_check.returncode == 0

LLAMA_DIR = Path("/content/llama.cpp")
if not LLAMA_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_DIR)],
        check=True,
    )

build_args = [
    "cmake", "-S", str(LLAMA_DIR), "-B", str(LLAMA_DIR / "build"),
    "-DCMAKE_BUILD_TYPE=Release",
    "-DGGML_CUDA=ON" if HAS_CUDA else "-DGGML_CUDA=OFF",
]
subprocess.run(build_args, check=True)
subprocess.run(
    ["cmake", "--build", str(LLAMA_DIR / "build"), "--config", "Release",
     "--target", "llama-server", "-j", "2"],
    check=True,
)

LLAMA_SERVER = LLAMA_DIR / "build" / "bin" / "llama-server"
if not LLAMA_SERVER.exists():
    raise FileNotFoundError(f"llama-server executable not found at {LLAMA_SERVER}")

print("Build complete:", LLAMA_SERVER)
print("CUDA enabled:", HAS_CUDA)

## Cell 4

In [ ]:
# Download Llama 3.2 3B Instruct in Q4_K_M GGUF format.
from huggingface_hub import hf_hub_download

MODEL_REPO = "bartowski/Llama-3.2-3B-Instruct-GGUF"
MODEL_FILE = "Llama-3.2-3B-Instruct-Q4_K_M.gguf"
MODEL_PATH = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir="/content/models/llama-3.2-3b",
)

print("Model:", MODEL_PATH)
print("Quantization: Q4_K_M")

## Cell 5

In [ ]:
# Launch llama.cpp's local OpenAI-compatible server.
SERVER_PORT = 8000
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}/v1"
SERVER_LOG_PATH = "/content/llama-server.log"

server_args = [
    str(LLAMA_SERVER),
    "-m", MODEL_PATH,
    "--host", "127.0.0.1",
    "--port", str(SERVER_PORT),
    "-c", "4096",
    "-a", "llama-3.2-3b-instruct",
    "-ngl", "99" if HAS_CUDA else "0",
]

server_log = open(SERVER_LOG_PATH, "w")
server_process = subprocess.Popen(
    server_args, stdout=server_log, stderr=subprocess.STDOUT
)

for _ in range(90):
    try:
        server_response = requests.get(f"{SERVER_URL}/models", timeout=2)
        if server_response.ok:
            break
    except requests.RequestException:
        pass
    if server_process.poll() is not None:
        raise RuntimeError(f"llama-server exited. Check {SERVER_LOG_PATH}")
    time.sleep(2)
else:
    raise TimeoutError(f"llama-server did not start. Check {SERVER_LOG_PATH}")

print("llama.cpp server is ready:", SERVER_URL)
print(server_response.json())

## Cell 6

In [ ]:
# Configure BeeAI to use the local OpenAI-compatible llama.cpp endpoint.
# The key is a placeholder; the local llama.cpp server does not require a real key.
os.environ["OPENAI_API_KEY"] = "sk-no-key-required"
os.environ["OPENAI_BASE_URL"] = SERVER_URL

from beeai_framework.agents.requirement import RequirementAgent
from beeai_framework.backend import ChatModel

MODEL_ID = "llama-3.2-3b-instruct"
llm = ChatModel.from_name(f"openai:{MODEL_ID}")

agent = RequirementAgent(
    llm=llm,
    role="AI Learning Assistant",
    instructions=(
        "Explain artificial intelligence concepts clearly to beginners. "
        "Be accurate, state uncertainty when appropriate, and use practical examples."
    ),
)

print("BeeAI agent configured for Llama 3.2 3B via llama.cpp.")

## Cell 7

In [ ]:
# Ask the agent a simple learning question.
response = await agent.run(
    "Explain what an AI agent is in approximately 100 words. "
    "Include one practical example."
)

print("========== BEEAI AGENT RESPONSE ==========\n")
print(response)

## Cell 8

In [ ]:
# Define a calculator tool with restricted arithmetic parsing.
# This avoids evaluating arbitrary Python expressions with eval().
import ast
import operator
from beeai_framework.tools import tool, StringToolOutput

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def safe_calculate(expression: str):
    tree = ast.parse(expression, mode="eval")

    def visit(node):
        if isinstance(node, ast.Expression):
            return visit(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
            left, right = visit(node.left), visit(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 10:
                raise ValueError("Exponent too large.")
            return _ALLOWED_OPERATORS[type(node.op)](left, right)
        if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
            return _ALLOWED_OPERATORS[type(node.op)](visit(node.operand))
        raise ValueError("Only basic arithmetic expressions are allowed.")

    return visit(tree)

@tool
def calculator(expression: str) -> StringToolOutput:
    'Calculate a basic arithmetic expression such as 23 * 47.'
    try:
        result = safe_calculate(expression)
        return StringToolOutput(result=str(result))
    except Exception as exc:
        return StringToolOutput(result=f"Could not calculate expression: {exc}")

print("Calculator tool created successfully.")

## Cell 9

In [ ]:
# Create a tool-enabled agent and ask it to calculate a result.
tool_agent = RequirementAgent(
    llm=llm,
    tools=[calculator],
    role="AI Learning Assistant",
    instructions=(
        "Explain concepts clearly. When a mathematical calculation is requested, "
        "use the calculator tool rather than calculating mentally."
    ),
)

tool_response = await tool_agent.run(
    "Use the calculator tool to calculate 23 * 47. Show the result."
)

print("========== TOOL-ENABLED AGENT ==========\n")
print(tool_response.last_message.text)

## Cell 10

In [ ]:
# Optional health check for the local model endpoint.
health = requests.get(f"{SERVER_URL}/models", timeout=10)
health.raise_for_status()
print("llama.cpp endpoint is healthy.")
print(health.json())

## Troubleshooting

- **Build errors:** Restart the Colab runtime and rerun setup. If CUDA compilation fails, set `HAS_CUDA = False` in Cell 3 and rebuild.
- **Model download fails:** Verify internet access and the Hugging Face repository/filename in Cell 4.
- **Server fails to start:** Read `/content/llama-server.log`. If GPU memory is insufficient, use a CPU runtime or reduce context size in Cell 5.
- **BeeAI cannot connect:** Confirm Cell 5 reports the server as ready and that `OPENAI_BASE_URL` in Cell 6 matches `SERVER_URL`.
- **Backend name differs in your BeeAI version:** This notebook uses `ChatModel.from_name("openai:llama-3.2-3b-instruct")` and standard `OPENAI_BASE_URL` / `OPENAI_API_KEY` variables. If your installed version uses different provider syntax, consult the BeeAI Framework documentation for that version.

This notebook has been structurally checked, but not executed end-to-end in a live Colab runtime.
